In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
BASE_DIR = Path.cwd().parent

DATA_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "resource_allocation_dataset.csv"
)
df = pd.read_csv(DATA_PATH)
df.head()

,Severity,Disaster_Type,People_Count,Children_Count,Elderly_Count,Injured_Count,Road_Accessibility,Weather_Condition,Location_Type,Time_Since_Incident,...,Water_Bottles,Temporary_Shelters,Supply_Trucks,Bulldozers,Excavators,Cranes,Power_Generators,Search_Dogs,Communication_Units,Fuel_Tankers
0,0.83,Flood,32,10,3,11,Accessible,Rain,Coastal,74,...,365,1,1,0,0,0,2,0,0,1
1,0.89,Flood,84,22,24,21,Not Accessible,Thunderstorm,Urban,85,...,1415,0,1,0,0,0,5,0,7,2
2,0.80,Earthquake,177,17,25,91,Not Accessible,Clear,Mountain,20,...,5009,4,4,2,3,0,6,5,4,5
3,0.62,Flood,41,18,16,11,Not Accessible,Heavy Rain,Urban,37,...,424,1,1,0,0,0,3,0,4,1
4,0.65,Fire,41,0,9,11,Not Accessible,Rain,Forest,32,...,453,1,1,0,0,0,0,0,0,1


In [3]:
print(df.shape)
df.info()
df.describe(include = "all")

(50000, 40)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 40 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Severity                 50000 non-null  float64
 1   Disaster_Type            50000 non-null  object 
 2   People_Count             50000 non-null  int64  
 3   Children_Count           50000 non-null  int64  
 4   Elderly_Count            50000 non-null  int64  
 5   Injured_Count            50000 non-null  int64  
 6   Road_Accessibility       50000 non-null  object 
 7   Weather_Condition        50000 non-null  object 
 8   Location_Type            50000 non-null  object 
 9   Time_Since_Incident      50000 non-null  int64  
 10  Incident_Time            50000 non-null  object 
 11  Infrastructure_Damage    50000 non-null  object 
 12  Communication_Available  50000 non-null  object 
 13  Evacuation_Required      50000 non-null  object 
 14  Priority_S

,Severity,Disaster_Type,People_Count,Children_Count,Elderly_Count,Injured_Count,Road_Accessibility,Weather_Condition,Location_Type,Time_Since_Incident,...,Water_Bottles,Temporary_Shelters,Supply_Trucks,Bulldozers,Excavators,Cranes,Power_Generators,Search_Dogs,Communication_Units,Fuel_Tankers
count,50000.000000,50000,50000.000000,50000.000000,50000.000000,50000.000000,50000,50000,50000,50000.000000,...,50000.0000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
unique,NaN,5,NaN,NaN,NaN,NaN,3,5,6,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,Flood,NaN,NaN,NaN,NaN,Not Accessible,Heavy Rain,Urban,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,17586,NaN,NaN,NaN,NaN,29901,12650,15244,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,0.780938,NaN,82.621640,10.943940,10.720740,22.937300,NaN,NaN,NaN,63.381020,...,1697.0434,1.686960,1.653100,0.376220,0.386700,0.225180,4.365980,0.688420,2.867340,1.917100
std,0.136210,NaN,56.995972,8.566055,8.674769,20.118625,NaN,NaN,NaN,44.472804,...,1763.8763,1.373144,1.167625,0.623479,0.678773,0.790194,2.821113,1.125808,2.893453,1.304723
min,0.400000,NaN,5.000000,0.000000,0.000000,1.000000,NaN,NaN,NaN,1.000000,...,26.0000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,0.690000,NaN,37.000000,4.000000,4.000000,9.000000,NaN,NaN,NaN,26.000000,...,416.0000,1.000000,1.000000,0.000000,0.000000,0.000000,2.000000,0.000000,0.000000,1.000000
50%,0.800000,NaN,68.000000,9.000000,8.000000,17.000000,NaN,NaN,NaN,55.000000,...,1097.0000,1.000000,1.000000,0.000000,0.000000,0.000000,4.000000,0.000000,2.000000,1.000000
75%,0.890000,NaN,123.000000,17.000000,17.000000,30.000000,NaN,NaN,NaN,96.000000,...,2174.0000,3.000000,2.000000,1.000000,1.000000,0.000000,6.000000,2.000000,5.000000,2.000000


In [4]:
df.isnull().sum()

Severity                   0
Disaster_Type              0
People_Count               0
Children_Count             0
Elderly_Count              0
Injured_Count              0
Road_Accessibility         0
Weather_Condition          0
Location_Type              0
Time_Since_Incident        0
Incident_Time              0
Infrastructure_Damage      0
Communication_Available    0
Evacuation_Required        0
Priority_Score             0
Priority                   0
Incident_ID                0
Ambulances                 0
Doctors                    0
Nurses                     0
Medical_Kits               0
Mobile_Medical_Units       0
Rescue_Teams               0
Fire_Trucks                0
Rescue_Boats               0
Helicopters                0
Rescue_Drones              0
Police                     0
Volunteers                 0
Food_Packets               0
Water_Bottles              0
Temporary_Shelters         0
Supply_Trucks              0
Bulldozers                 0
Excavators    

In [5]:
df_encoded = pd.get_dummies(
    df,
    columns=[
        "Disaster_Type",
        "Road_Accessibility",
        "Weather_Condition",
        "Location_Type",
        "Incident_Time",
        "Infrastructure_Damage",
        "Communication_Available",
        "Evacuation_Required",
        "Priority"
    ]
)
df_encoded.head()

,Severity,People_Count,Children_Count,Elderly_Count,Injured_Count,Time_Since_Incident,Priority_Score,Incident_ID,Ambulances,Doctors,...,Infrastructure_Damage_Low,Infrastructure_Damage_Medium,Communication_Available_No,Communication_Available_Yes,Evacuation_Required_No,Evacuation_Required_Yes,Priority_Critical,Priority_High,Priority_Low,Priority_Medium
0,0.83,32,10,3,11,74,40.88,INC-000001,3,3,...,False,True,False,True,False,True,False,False,False,True
1,0.89,84,22,24,21,85,71.88,INC-000002,8,6,...,False,False,True,False,True,False,False,True,False,False
2,0.80,177,17,25,91,20,79.48,INC-000003,41,22,...,False,True,True,False,False,True,True,False,False,False
3,0.62,41,18,16,11,37,49.68,INC-000004,4,2,...,False,True,True,False,False,True,False,False,False,True
4,0.65,41,0,9,11,32,39.37,INC-000005,3,2,...,True,False,False,True,False,True,False,False,False,True


In [6]:
df_encoded.columns

Index(['Severity', 'People_Count', 'Children_Count', 'Elderly_Count',
       'Injured_Count', 'Time_Since_Incident', 'Priority_Score', 'Incident_ID',
       'Ambulances', 'Doctors', 'Nurses', 'Medical_Kits',
       'Mobile_Medical_Units', 'Rescue_Teams', 'Fire_Trucks', 'Rescue_Boats',
       'Helicopters', 'Rescue_Drones', 'Police', 'Volunteers', 'Food_Packets',
       'Water_Bottles', 'Temporary_Shelters', 'Supply_Trucks', 'Bulldozers',
       'Excavators', 'Cranes', 'Power_Generators', 'Search_Dogs',
       'Communication_Units', 'Fuel_Tankers', 'Disaster_Type_Cyclone',
       'Disaster_Type_Earthquake', 'Disaster_Type_Fire', 'Disaster_Type_Flood',
       'Disaster_Type_Landslide', 'Road_Accessibility_Accessible',
       'Road_Accessibility_Not Accessible',
       'Road_Accessibility_Partially Accessible', 'Weather_Condition_Clear',
       'Weather_Condition_Cyclone', 'Weather_Condition_Heavy Rain',
       'Weather_Condition_Rain', 'Weather_Condition_Thunderstorm',
       'Location_T

In [7]:
target_columns = [
    # Medical
    "Ambulances",
    "Doctors",
    "Nurses",
    "Medical_Kits",
    "Mobile_Medical_Units",

    # Rescue
    "Rescue_Teams",
    "Fire_Trucks",
    "Rescue_Boats",
    "Helicopters",
    "Rescue_Drones",

    # Personnel
    "Police",
    "Volunteers",

    # Relief
    "Food_Packets",
    "Water_Bottles",
    "Temporary_Shelters",
    "Supply_Trucks",

    # Heavy Equipment
    "Bulldozers",
    "Excavators",
    "Cranes",

    # Infrastructures
    "Power_Generators",
    "Communication_Units",
    "Fuel_Tankers",

    # Search & Rescue
    "Search_Dogs",

]
X=df_encoded.drop(
    columns = ["Incident_ID"] + target_columns
)

y  = df_encoded[target_columns]

print("Feature Shape :", X.shape)
print("Target Shape :", y.shape)




Feature Shape : (50000, 41)
Target Shape : (50000, 23)


In [8]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(40000, 41)
(10000, 41)
(40000, 23)
(10000, 23)


In [9]:
scaler=StandardScaler()
numeric_columns=["Severity","People_Count","Children_Count","Elderly_Count","Time_Since_Incident","Priority_Score"]
X_train[numeric_columns]=scaler.fit_transform(X_train[numeric_columns])
X_test[numeric_columns]=scaler.fit_transform(X_test[numeric_columns])

In [10]:
rf_model=MultiOutputRegressor(RandomForestRegressor(n_estimators=50,max_depth=15,random_state=42,n_jobs=-1))
rf_model.fit(X_train,y_train)
rf_predictions=rf_model.predict(X_test)
rf_r2=r2_score(y_test,rf_predictions,multioutput="uniform_average")
print(round(rf_r2,4))

0.949


In [11]:
et_model=MultiOutputRegressor(ExtraTreesRegressor(n_estimators=100,max_depth=15,random_state=42,n_jobs=-1))
et_model.fit(X_train,y_train)
et_predictions=et_model.predict(X_test)
et_r2=r2_score(y_test,et_predictions,multioutput="uniform_average")
print(round(et_r2,4))

0.9491


In [12]:
xgb_model=MultiOutputRegressor(XGBRegressor(n_estimators=100,max_depth=6,learning_rate=0.1,random_state=42,n_jobs=-1,objective="reg:squarederror"))
xgb_model.fit(X_train,y_train)
xgb_predictions = xgb_model.predict(X_test)
xgb_r2 = r2_score(y_test, xgb_predictions, multioutput = "uniform_average")
print("XGBoost R2 score :",round(xgb_r2,4))

XGBoost R2 score : 0.9474


In [14]:
comparison = pd.DataFrame({
    "Model":[
        "Random Forest",
        "Extra Trees",
        "XGBoost"
    ],
    "R2 Score":[
        rf_r2,
        et_r2,
        xgb_r2
    ]
})
comparison = comparison.sort_values(by="R2 Score",ascending=False)
comparison

,Model,R2 Score
1,Extra Trees,0.949122
0,Random Forest,0.948993
2,XGBoost,0.947378


In [16]:
import joblib
from pathlib import Path
MODEL_DIR=Path.cwd().parent/"models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
joblib.dump(et_model,MODEL_DIR/"resource_allocation_model.pkl")
joblib.dump(scaler,MODEL_DIR/"resource_scaler.pkl")
joblib.dump(X.columns.tolist(),MODEL_DIR/"resource_feature_columns.pkl")
print("Done saving this model")

Done saving this model


In [18]:
print(list(MODEL_DIR.iterdir()))

[WindowsPath('C:/Users/HP/n1/ResqAI/models/.gitkeep'), WindowsPath('C:/Users/HP/n1/ResqAI/models/feature_columns.json'), WindowsPath('C:/Users/HP/n1/ResqAI/models/label_encoders'), WindowsPath('C:/Users/HP/n1/ResqAI/models/priority_feature_column.json'), WindowsPath('C:/Users/HP/n1/ResqAI/models/priority_label_encoder.pkl'), WindowsPath('C:/Users/HP/n1/ResqAI/models/priority_model.pkl'), WindowsPath('C:/Users/HP/n1/ResqAI/models/priority_prediction'), WindowsPath('C:/Users/HP/n1/ResqAI/models/priority_scaler.pkl'), WindowsPath('C:/Users/HP/n1/ResqAI/models/priority_training_report.json'), WindowsPath('C:/Users/HP/n1/ResqAI/models/reports'), WindowsPath('C:/Users/HP/n1/ResqAI/models/resource_allocation_model.pkl'), WindowsPath('C:/Users/HP/n1/ResqAI/models/resource_feature_columns.pkl'), WindowsPath('C:/Users/HP/n1/ResqAI/models/resource_scaler.pkl'), WindowsPath('C:/Users/HP/n1/ResqAI/models/scalers'), WindowsPath('C:/Users/HP/n1/ResqAI/models/trained_models')]
